# Static Word Embeddings

This notebook explores pretrained static word embeddings using
the adapted conversational corpus introduced in Notebook 1.

Unlike Bag-of-Words and TF-IDF, static embeddings represent words
as dense numerical vectors learned from patterns of word co-occurrence.

The analysis examines vocabulary coverage, lexical similarity,
and the treatment of novel linguistic expressions, particularly
*frankenstein* and *frankensteining*.

The objective is to demonstrate the application and limitations
of pretrained static embeddings, rather than train a model on
the small project corpus.

## 1. Loading the pretrained model

We use spaCy's `en_core_web_md` English pipeline, which provides
300-dimensional static word vectors.

The model was pretrained on external text data. No additional
training is performed on our small conversational corpus.

In [1]:
# Check whether there are vectors for target terms.

import spacy

nlp = spacy.load("en_core_web_md")

target_words = ["frankenstein", "frankensteining"]

for word in target_words:
    token = nlp(word)[0]

    print(
        word,
        "| Has vector:", token.has_vector,
        "| Vector norm:", round(token.vector_norm, 4)
    )

frankenstein | Has vector: True | Vector norm: 6.3077
frankensteining | Has vector: False | Vector norm: 0.0


In [2]:
for word in target_words:
    doc = nlp(word)
    print(word, "→", [token.text for token in doc])

frankenstein → ['frankenstein']
frankensteining → ['frankensteining']


In [3]:
# Comparison words to check whether associations with the novel are reflected.

reference_word = nlp("frankenstein")[0]

comparison_words = [
    "monster",
    "horror",
    "creature",
    "reconstruction",
    "combination",
    "modification"
]

for word in comparison_words:
    token = nlp(word)[0]

    print(
        word,
        "| Similarity:",
        round(reference_word.similarity(token), 4)
    )

monster | Similarity: 0.137
horror | Similarity: 0.0981
creature | Similarity: 0.0559
reconstruction | Similarity: 0.0992
combination | Similarity: 0.0555
modification | Similarity: 0.0571


None of the six comparison words is strongly aligned with the pretrained representation of frankenstein.

In [4]:
# Check capitalization

for word in ["frankenstein", "Frankenstein"]:
    token = nlp(word)[0]

    print(
        word,
        "| Has vector:", token.has_vector,
        "| Vector norm:", round(token.vector_norm, 4),
        "| Similarity to monster:",
        round(token.similarity(nlp("monster")[0]), 4)
    )

frankenstein | Has vector: True | Vector norm: 6.3077 | Similarity to monster: 0.137
Frankenstein | Has vector: True | Vector norm: 6.3077 | Similarity to monster: 0.137


Both vectors have the same vector norm and similarity to "monster"; so they are being treated identically at the vector level.

In [5]:
# Find 5 most similar word or 5 nearest neighbours

import numpy as np

# Get the vector for our target word
target_vector = nlp("frankenstein")[0].vector

# Find the five nearest vectors
keys, _, scores = nlp.vocab.vectors.most_similar(
    np.asarray([target_vector], dtype="float32"),
    n=5
)

# Display the results
for key, score in zip(keys[0], scores[0]):
    print(nlp.vocab.strings[int(key)], round(float(score), 4))

Terrance 1.0
eastwood 1.0
Roosevelt 1.0
julius 0.7495
HEATH 0.7495


In [6]:
for word in ["frankenstein", "Terrance", "eastwood", "Roosevelt"]:
    token = nlp(word)[0]

    print(
        word,
        "| Has vector:", token.has_vector,
        "| Same vector:",
        np.array_equal(target_vector, token.vector)
    )

frankenstein | Has vector: True | Same vector: True
Terrance | Has vector: True | Same vector: True
eastwood | Has vector: True | Same vector: True
Roosevelt | Has vector: True | Same vector: True


In [7]:
for word in ["frankenstein", "Terrance", "eastwood", "Roosevelt"]:
    lexeme = nlp.vocab[word]

    print(
        word,
        "| Vector row:",
        lexeme.rank
    )

frankenstein | Vector row: 14630
Terrance | Vector row: 14630
eastwood | Vector row: 14629
Roosevelt | Vector row: 14628


In [8]:
for word in ["frankenstein", "Terrance", "eastwood", "Roosevelt"]:
    lexeme = nlp.vocab[word]
    row = nlp.vocab.vectors.key2row.get(lexeme.orth)

    print(word, "| Actual vector row:", row)

frankenstein | Actual vector row: 14630
Terrance | Actual vector row: 14630
eastwood | Actual vector row: 14629
Roosevelt | Actual vector row: 14628


## 2. Vocabulary coverage and lexical similarity

The pretrained `en_core_web_md` model provides 300-dimensional static word vectors.

Both target expressions, *frankenstein* and *frankensteining*, were successfully tokenized as single tokens. However, their vector coverage differed:

- **frankenstein:** A pretrained vector was available (vector norm: 6.3077).
- **frankensteining:** No pretrained vector was available (vector norm: 0.0).

This illustrates an out-of-vocabulary limitation: a model may recognize an expression as a token without providing a pretrained lexical vector for it.

### Lexical similarity

Cosine similarity comparisons between *frankenstein* and six selected words produced relatively low values. The highest similarity was observed with *monster* (0.1370), followed by *reconstruction* (0.0992) and *horror* (0.0981).

These comparisons did not establish a clear semantic association with either the literary meaning of *Frankenstein* or the meaning developed in the conversational corpus.

A subsequent nearest-neighbour search revealed that *frankenstein*, *Terrance*, *eastwood* and *Roosevelt* shared identical numerical vectors.

Inspection of spaCy's vector table confirmed that *frankenstein* and *Terrance* mapped to the same vector row (14630), while *eastwood* and *Roosevelt* mapped to different rows containing identical vectors.

This result is consistent with the effects of vector-table compression or pruning, where distinct vocabulary entries may share numerical representations.

### Interpretation

The findings illustrate two limitations of this pretrained static embedding model:

1. **Vocabulary coverage:** Novel lexical formations such as *frankensteining* may lack pretrained vectors.
2. **Lexical differentiation:** Having a vector does not guarantee a unique or semantically distinctive representation.

Furthermore, static word vectors do not adapt to the conversational context in which a word appears.

Consequently, the available representation of *frankenstein* cannot independently capture the particular meaning established through its use in the analysed exchanges.

These observations motivate the examination of contextual embeddings in Notebook 4.